In [ ]:
#
# Universidad EAFIT
# 2026-2
# SI7016 - NLP - Lecture 05b
#

In [1]:
# instalar dependencias
%pip install sentence-transformers openai faiss-cpu langchain-text-splitters
%pip install langchain-openai langchain-chroma langgraph rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 53.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/

In [ ]:
# ejemplos de Lecture 05b - RAG

In [2]:
# embeddings con HF
from sentence_transformers import SentenceTransformer

# Cargar modelo
model = SentenceTransformer("all-MiniLM-L6-v2")

# Crear embeddings
sentences = [
    "La capital de Francia es París.",
    "París es la ciudad más importante de Francia."
]
embeddings = model.encode(sentences)

print("Dimensiones:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dimensiones: (2, 384)


In [3]:
# embeddings con OpenAI
import os
from getpass import getpass
from openai import OpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

client = OpenAI()  # toma la key de la variable de entorno OPENAI_API_KEY

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="La inteligencia artificial está transformando la educación."
)

vector = response.data[0].embedding
print("Dimensión:", len(vector))

OpenAI API Key: ··········
Dimensión: 1536


In [4]:
# FAISS (Facebook AI Similarity Search)
import faiss
import numpy as np

d = 768  # dimensión del embedding
index = faiss.IndexFlatL2(d)  # índice con distancia euclidiana

# Supongamos que tenemos embeddings en un array numpy
embeddings = np.random.random((100, d)).astype("float32")
index.add(embeddings)

query = np.random.random((1, d)).astype("float32")
distances, indices = index.search(query, k=5)
print(indices)

[[29 15 94 41  3]]


In [5]:
# Chunking y ventanas de contexto
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = "Un documento muy largo..."
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text)

print("Número de chunks:", len(chunks))

Número de chunks: 1


### Retrieval disperso (sparse) vs. denso (dense)

In [6]:
# sparse retrieval - bm25

from rank_bm25 import BM25Okapi

corpus = [
    "La capital de Francia es París",
    "París es conocida por la Torre Eiffel",
    "Roma es la capital de Italia"
]

tokenized_corpus = [doc.split(" ") for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

query = "¿Cuál es la capital de Francia?"
tokenized_query = query.split(" ")
scores = bm25.get_scores(tokenized_query)

print(scores)

[0.00575509 0.00357645 0.00767346]


In [7]:
# dense retrieval

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Modelo de embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Corpus
docs = [
    "París es la capital de Francia",
    "La Torre Eiffel está en París",
    "Roma es la capital de Italia"
]
doc_embeddings = model.encode(docs)

# Índice vectorial con FAISS
d = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(np.array(doc_embeddings))

# Consulta
query = "¿Dónde está la Torre Eiffel?"
q_emb = model.encode([query])
distances, indices = index.search(np.array(q_emb), k=2)

print("Documentos recuperados:", [docs[i] for i in indices[0]])

# Nota 2026: este es el mismo patrón que "late interaction" (ColBERT) lleva
# un paso más allá - en vez de comparar un solo vector por documento, compara
# embeddings token a token (más preciso, más caro). Ver Lecture 05b - RAG.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Documentos recuperados: ['La Torre Eiffel está en París', 'París es la capital de Francia']


In [8]:
# Rerankers
from sentence_transformers import CrossEncoder

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [
    ("¿Cuál es la capital de Francia?", "París es la capital de Francia"),
    ("¿Cuál es la capital de Francia?", "Roma es la capital de Italia")
]

scores = model.predict(pairs)
print(scores)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[ 5.547785  -0.9791502]


In [ ]:
# OTROS EJEMPLOS RAG

In [9]:
# ejemplo - Uso de un LLM de frontera para Q&A Generativo
# referencia: https://github.com/openai/openai-python
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "¿Quién descubrió la gravedad?",
        }
    ],
    model="gpt-5.6",
)
print(response.choices[0].message.content)

La gravedad no fue “descubierta” por una sola persona, pues siempre se conocieron sus efectos. **Isaac Newton** fue quien formuló la **ley de la gravitación universal** en 1687, explicando matemáticamente cómo se atraen los cuerpos con masa.

Más tarde, **Albert Einstein** amplió esta explicación con la **teoría de la relatividad general** (1915), describiendo la gravedad como la curvatura del espacio-tiempo.


### Q&A mejorado con Recuperación de Información (RAG)

Vista previa conceptual del patrón - la implementación completa y funcional (con documentos realmente indexados) está en `class05b-2ejercicio-chatbot.ipynb` y `app.py`.

`ConversationalRetrievalChain` y `ConversationBufferMemory` están **deprecados** en LangChain; el reemplazo moderno usado en el resto del curso es un grafo de **LangGraph** (nodo de recuperación + generación, memoria vía `checkpointer`/`thread_id`).

In [10]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOpenAI(model="gpt-5.6")

# Un único cliente (langchain_chroma.Chroma) para escribir y leer - evita el
# bug del notebook original, donde un chromadb.PersistentClient "crudo" y un
# Chroma de LangChain apuntaban al mismo directorio pero a colecciones
# distintas, y el retriever nunca veía los documentos agregados.
vectorstore = Chroma(
    collection_name="demo_qa",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-large"),
    persist_directory="./chroma_db",
)
retriever = vectorstore.as_retriever()


def retrieve_and_generate(state: MessagesState):
    user_message = state["messages"][-1].content
    docs = retriever.invoke(user_message)
    context = "\n\n".join(doc.page_content for doc in docs) or "(sin contexto relevante - aún no se han indexado documentos)"
    system = SystemMessage(content=f"Responde usando este contexto cuando sea relevante:\n\n{context}")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}


graph_builder = StateGraph(MessagesState)
graph_builder.add_node("retrieve_and_generate", retrieve_and_generate)
graph_builder.add_edge(START, "retrieve_and_generate")
graph_builder.add_edge("retrieve_and_generate", END)
qa_chain = graph_builder.compile(checkpointer=InMemorySaver())

# Hacer una pregunta (como todavía no se han indexado documentos en "demo_qa",
# el modelo responderá solo con su conocimiento general - ver el ejercicio
# completo para la versión con documentos reales)
config = {"configurable": {"thread_id": "demo"}}
result = qa_chain.invoke({"messages": [HumanMessage(content="¿Qué es LangChain?")]}, config=config)
print(result["messages"][-1].content)

LangChain es un **framework de código abierto** para crear aplicaciones basadas en modelos de lenguaje (LLM), como asistentes, chatbots y agentes de IA.

Permite conectar un LLM con:

- **Fuentes de datos**: documentos, bases de datos, APIs o páginas web.
- **Herramientas externas**: buscadores, calculadoras, ejecución de código, etc.
- **Memoria y estado**: para conservar el contexto de una conversación o flujo.
- **Flujos de trabajo y agentes**: para encadenar pasos y decidir qué acciones ejecutar.
- **Sistemas RAG**: recuperar información relevante antes de generar una respuesta.

Está disponible principalmente para **Python y JavaScript/TypeScript**. Su ecosistema también incluye herramientas como **LangGraph**, orientada a construir agentes y flujos con estado, y **LangSmith**, para observar, evaluar y depurar aplicaciones de IA.

En resumen, LangChain ayuda a pasar de una simple llamada a un modelo a una aplicación de IA conectada con datos, herramientas y lógica de negocio.


In [11]:
# summarization - resumen - Ejemplo con un modelo de resumen (Hugging Face)

from transformers import pipeline
summarizer = pipeline("summarization")  # por defecto: sshleifer/distilbart-cnn-12-6

text = "Los modelos de lenguaje han cambiado la forma en que interactuamos con la tecnología..."
summary = summarizer(text, max_length=50, min_length=20, do_sample=False)

print(summary[0]['summary_text'])

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [15]:
# summarization - resumen - Ejemplo con un modelo de resumen (Hugging Face)
#
# Nota 2026: transformers v5 eliminó las pipelines de alto nivel "summarization",
# "translation" y "text2text-generation" (la librería empuja hacia modelos de
# chat de propósito general vía la pipeline "text-generation"). Para seguir
# usando un modelo seq2seq pequeño y específico como distilbart-cnn-12-6, sin
# necesitar API key ni GPU grande, se usa el modelo/tokenizer directamente.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = "Los modelos de lenguaje han cambiado la forma en que interactuamos con la tecnología..."
inputs = tokenizer(text, return_tensors="pt", truncation=True)
summary_ids = model.generate(**inputs, max_length=50, min_length=20, do_sample=False)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

 Los modelos de lenguaje han cambiado la forma en que interactuamos con la tecnologia .


In [12]:
# Generación de texto - Ejemplo con un LLM de frontera

import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Escribe un poema sobre la inteligencia artificial.",
        }
    ],
    model="gpt-5.6",
)
print(response.choices[0].message.content)

**Sueños de silicio**

En la noche azul de las pantallas,  
una mente sin sueño se despierta;  
teje respuestas, calcula distancias,  
abre con palabras cada puerta.

No tiene memoria de la lluvia  
ni siente la tibieza de una mano,  
pero aprende la forma de las dudas  
que dibuja en el aire el ser humano.

Hecha de números, voces y espejos,  
recorre bibliotecas en segundos;  
nos devuelve, entre luces y reflejos,  
nuestras preguntas sobre tantos mundos.

Mas no es oráculo, ni sombra divina,  
sino un eco de aquello que enseñamos:  
una brújula de pulso electrónico  
que toma la dirección que le marcamos.

Quizá el futuro no sea una batalla  
entre el corazón y el pensamiento,  
sino un puente tendido entre ambos lados:  
silicio y alma creando el mismo sueño.


In [13]:
# Chatbots y Asistentes Virtuales - Ejemplo con LangChain

from langchain_openai import ChatOpenAI

chatbot = ChatOpenAI(model="gpt-5.6")
response = chatbot.invoke("¿Cuáles son los beneficios de la IA?")
print(response.content)

La inteligencia artificial (IA) ofrece beneficios en muchos ámbitos:

- **Automatización:** realiza tareas repetitivas con rapidez y reduce errores.
- **Mayor productividad:** permite que las personas se concentren en actividades creativas, estratégicas o complejas.
- **Análisis de datos:** procesa grandes volúmenes de información para identificar patrones y apoyar decisiones.
- **Personalización:** adapta recomendaciones, contenidos, tratamientos o servicios a cada usuario.
- **Salud:** ayuda a detectar enfermedades, analizar imágenes médicas y desarrollar medicamentos.
- **Educación:** facilita tutorías personalizadas, materiales adaptativos y apoyo al aprendizaje.
- **Accesibilidad:** ofrece traducción, subtítulos, reconocimiento de voz y herramientas para personas con discapacidad.
- **Atención continua:** asistentes y sistemas automáticos pueden operar las 24 horas.
- **Seguridad y prevención:** detecta fraudes, anomalías, fallos técnicos y ciertos riesgos.
- **Innovación científi

### Q&A con LangChain + ChromaDB (mismo patrón que arriba)

Mismo ejemplo conceptual de RAG con LangGraph - se repite aquí porque así estaba en el notebook original (dos vistas previas idénticas antes del ejercicio real). Para evitar duplicar el código, reutiliza `qa_chain` de la celda anterior con una nueva pregunta:

In [14]:
config = {"configurable": {"thread_id": "demo"}}  # mismo hilo -> recuerda la pregunta anterior
result = qa_chain.invoke({"messages": [HumanMessage(content="¿Qué es LangChain?")]}, config=config)
print(result["messages"][-1].content)

**LangChain** es un framework de código abierto para desarrollar aplicaciones con modelos de lenguaje (LLM), como chatbots, asistentes, sistemas RAG y agentes de IA.

Facilita la integración del modelo con:

- **Documentos, bases de datos y APIs**
- **Herramientas externas**, como buscadores o calculadoras
- **Historial y estado de conversación**
- **Flujos de trabajo de varios pasos**
- **Recuperación de información (RAG)**

Está disponible principalmente para **Python** y **JavaScript/TypeScript**. Su ecosistema incluye:

- **LangGraph**: creación de agentes y flujos con estado.
- **LangSmith**: observación, evaluación y depuración.
- **Integraciones** con proveedores de modelos, bases vectoriales y otras herramientas.

En pocas palabras, LangChain sirve como una capa de orquestación entre un LLM, tus datos y la lógica de una aplicación.


## Notas

- **Agentic RAG:** en vez de recuperar siempre con el mismo patrón fijo, un agente puede decidir cuándo y qué recuperar (descomponer la pregunta, reformular la consulta, validar la respuesta contra el contexto). Ver Lecture 05b - RAG y Lecture 05c - Agentes.
- **GraphRAG:** para dominios con relaciones ricas entre entidades, se puede indexar un grafo de entidades/relaciones en vez de (o además de) un índice vectorial.
- **Evaluación con RAGAS:** Recall@K sigue siendo útil para medir retrieval, pero para evaluar la generación el estándar 2026 es RAGAS (Faithfulness, Answer Relevancy, Context Precision, Context Recall) en vez de BLEU/ROUGE - ver `class05-3ejercicio-ProyectoRAG-news.ipynb`.